# SkinScan AI — NLP Symptoms Module (Member 4)
## BioBERT-only training pipeline — 10 classes, English

Single self-contained notebook. Run top to bottom (**Runtime > Run all**).
Cell 2 will ask you to upload `symptom_dataset_v2.csv`.

**Outputs at the end (for handoff to Members 2 and 5):**
- `symptom_classifier_biobert.pkl` — trained model
- `interface_metadata.json` — embedding dimension, condition list, triage mapping
- `evaluation_report.txt` — test-set + val-set metrics, for your documentation

## Step 0 — Install dependencies
`torch` is preinstalled on Colab. `tokenizers` and `huggingface_hub` are core transformers dependencies (always present) — using them directly avoids a known BioBERT tokenizer conversion bug in `AutoTokenizer`.

In [ ]:
!pip install transformers tokenizers huggingface_hub -q

## Step 1 — Upload the dataset

In [ ]:
from google.colab import files
print("Upload symptom_dataset_v2.csv")
uploaded = files.upload()

## Step 2 — Preprocessing

In [ ]:
import re

_PUNCT_PATTERN = re.compile(r'[^\w\s]', flags=re.UNICODE)
_MULTI_SPACE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = _PUNCT_PATTERN.sub(' ', text)
    text = _MULTI_SPACE.sub(' ', text).strip()
    return text

def clean_batch(texts):
    return [clean_text(t) for t in texts]

print("preprocessing ready")

## Step 3 — Severity scoring (rule-based)
Same logic used to label the synthetic data — kept identical so training labels and inference-time scoring never drift apart.

In [ ]:
HIGH_RISK_SYMPTOMS = {
    "bleeding spot", "changing mole", "new dark lesion", "irregular border",
    "asymmetric mole", "uneven mole color", "mole growing larger",
    "sore that won't heal", "sore that bleeds and returns",
}

TRIAGE_MAP = {
    "mild": "Home Care",
    "moderate": "Home Care",
    "severe": "See a Doctor",
    "urgent": "URGENT",
}

def severity_label(score: float) -> str:
    if score >= 0.8:
        return "urgent"
    elif score >= 0.55:
        return "severe"
    elif score >= 0.35:
        return "moderate"
    else:
        return "mild"

def severity_score(symptoms: dict) -> dict:
    base = symptoms.get("base_severity", 0.3)
    duration = symptoms.get("duration_days", 0) or 0
    spreading = bool(symptoms.get("spreading", False))
    pain = symptoms.get("pain_level", "none")
    itch = symptoms.get("itch_level", "none")
    tags = set(symptoms.get("symptom_type", []) or [])

    score = base
    if duration > 45:
        score += 0.08
    if spreading:
        score += 0.12
    if pain in ("moderate", "severe"):
        score += 0.08
    if itch == "severe":
        score += 0.04
    if tags & HIGH_RISK_SYMPTOMS:
        score = max(score, 0.66)

    score = round(max(0.0, min(score, 1.0)), 2)
    label = severity_label(score)
    return {"severity_score": score, "severity_label": label, "triage_level": TRIAGE_MAP[label]}

print("severity scoring ready")

## Step 4 — BioBERT embeddings (fixed loading path)

**Why this approach:** `AutoTokenizer.from_pretrained` tries to build a "fast" tokenizer and
raises `ValueError: ...need sentencepiece or tiktoken...` for this particular BioBERT repo,
even though BioBERT uses plain WordPiece (no sentencepiece involved at all) — it's a
conversion-path bug, not a real dependency gap. Installing `sentencepiece` sometimes needs a
runtime restart to take effect, which breaks a "Run All" flow.

Instead, this loads BioBERT's `vocab.txt` directly via the `tokenizers` library's
`BertWordPieceTokenizer` — the same WordPiece algorithm, no conversion step, no restart needed.

In [ ]:
import torch
from transformers import AutoModel
from tokenizers import BertWordPieceTokenizer
from huggingface_hub import hf_hub_download

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
EMBEDDING_DIM = 768
MAX_LENGTH = 128

_tokenizer = None
_model = None

def _load_biobert():
    global _tokenizer, _model
    if _model is None:
        print("Downloading BioBERT vocab + weights (first run only, ~440MB)...")
        vocab_path = hf_hub_download(repo_id=MODEL_NAME, filename="vocab.txt")
        _tokenizer = BertWordPieceTokenizer(vocab_path, lowercase=False)
        _tokenizer.enable_truncation(max_length=MAX_LENGTH)
        _model = AutoModel.from_pretrained(MODEL_NAME)
        _model.eval()
        print("BioBERT loaded.")
    return _tokenizer, _model

def _mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def _encode_batch(texts, tokenizer):
    tokenizer.enable_padding(length=None)  # pad to longest in this batch
    encodings = tokenizer.encode_batch(texts)
    max_len = max(len(e.ids) for e in encodings)
    input_ids = torch.tensor([e.ids + [0] * (max_len - len(e.ids)) for e in encodings])
    attention_mask = torch.tensor([e.attention_mask + [0] * (max_len - len(e.attention_mask)) for e in encodings])
    return input_ids, attention_mask

@torch.no_grad()
def embed_text(text: str) -> list:
    tokenizer, model = _load_biobert()
    input_ids, attention_mask = _encode_batch([text], tokenizer)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    pooled = _mean_pool(outputs.last_hidden_state, attention_mask)
    return pooled.squeeze(0).tolist()

@torch.no_grad()
def embed_batch(texts: list, batch_size: int = 16) -> list:
    tokenizer, model = _load_biobert()
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        input_ids, attention_mask = _encode_batch(batch, tokenizer)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = _mean_pool(outputs.last_hidden_state, attention_mask)
        all_embeddings.extend(pooled.tolist())
        print(f"  embedded {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
    print()
    return all_embeddings

test_vec = embed_text("I have a pearly bump on my ear that won't heal")
print(f"BioBERT working. Embedding dim: {len(test_vec)}")

## Step 5 — Stratified train/val/test split (70/15/15)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

df = pd.read_csv("symptom_dataset_v2.csv", encoding="utf-8-sig")

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=RANDOM_STATE, stratify=df["condition"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_df["condition"]
)

print(f"Total: {len(df)} | Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df["condition"].value_counts())

CONDITION_LIST = sorted(df["condition"].unique().tolist())
print(f"\n{len(CONDITION_LIST)} classes:")
for c in CONDITION_LIST:
    print(f"  - {c}")

## Step 6 — Extract BioBERT embeddings for train/val/test
This is the slow step — expect a few minutes for ~2000 rows on Colab's free CPU/GPU.

In [ ]:
print("Embedding train set...")
X_train = embed_batch(clean_batch(train_df["text"]))
print("Embedding val set...")
X_val = embed_batch(clean_batch(val_df["text"]))
print("Embedding test set...")
X_test = embed_batch(clean_batch(test_df["text"]))

y_train = train_df["condition"]
y_val = val_df["condition"]
y_test = test_df["condition"]

print(f"\nDone. {len(X_train)} train / {len(X_val)} val / {len(X_test)} test embeddings, dim={len(X_train[0])}")

## Step 7 — Train the classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Validation accuracy: {val_acc:.3f}\n")
print(classification_report(y_val, val_preds))

## Step 8 — Final evaluation on the held-out test set

In [ ]:
test_preds = model.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)
test_report = classification_report(y_test, test_preds)

print(f"Test accuracy: {test_acc:.3f}\n")
print(test_report)

from sklearn.metrics import confusion_matrix
labels_sorted = sorted(y_test.unique())
cm = confusion_matrix(y_test, test_preds, labels=labels_sorted)
print("Confusion matrix (rows=true, cols=predicted):")
print("Labels order:", labels_sorted)
print(cm)

## Step 9 — Save the model and interface metadata
These are the exact files to hand to Members 2 and 5.

In [ ]:
import joblib
import json

MODEL_OUT = "symptom_classifier_biobert.pkl"
joblib.dump({
    "model": model,
    "embedding_dim": EMBEDDING_DIM,
    "condition_list": CONDITION_LIST,
}, MODEL_OUT)
print(f"Saved -> {MODEL_OUT}")

metadata = {
    "text_embedding_dim": EMBEDDING_DIM,
    "embedding_source": "dmis-lab/biobert-base-cased-v1.1 (mean-pooled)",
    "language_support": "English only",
    "condition_list": CONDITION_LIST,
    "severity_labels": ["mild", "moderate", "severe", "urgent"],
    "triage_map": TRIAGE_MAP,
    "test_accuracy": round(test_acc, 3),
    "val_accuracy": round(val_acc, 3),
}
with open("interface_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved -> interface_metadata.json")

with open("evaluation_report.txt", "w") as f:
    f.write(f"Validation accuracy: {val_acc:.3f}\n\n")
    f.write(classification_report(y_val, val_preds))
    f.write(f"\n\nTest accuracy: {test_acc:.3f}\n\n")
    f.write(test_report)
print("Saved -> evaluation_report.txt")

## Step 10 — Sanity check: full inference on a real input

In [ ]:
CONDITION_BASE_SEVERITY = {
    "Eczema": 0.30,
    "Melanoma": 0.85,
    "Atopic Dermatitis": 0.40,
    "Basal Cell Carcinoma (BCC)": 0.60,
    "Melanocytic Nevi (NV)": 0.10,
    "Benign Keratosis-like Lesions (BKL)": 0.25,
    "Psoriasis pictures Lichen Planus and related diseases": 0.45,
    "Seborrheic Keratoses and other Benign Tumors": 0.20,
    "Tinea Ringworm Candidiasis and other Fungal Infections": 0.20,
    "Warts Molluscum and other Viral Infections": 0.15,
}

def predict(raw_text: str, symptom_fields: dict = None) -> dict:
    cleaned = clean_text(raw_text)
    embedding = embed_text(cleaned)
    condition = model.predict([embedding])[0]

    fields = dict(symptom_fields or {})
    fields.setdefault("base_severity", CONDITION_BASE_SEVERITY.get(condition, 0.3))
    sev = severity_score(fields)

    return {
        "condition": condition,
        "severity_score": sev["severity_score"],
        "severity_label": sev["severity_label"],
        "triage_level": sev["triage_level"],
        "text_embedding": embedding,
    }

result = predict(
    "I noticed a pearly bump on my ear that keeps bleeding and won't heal",
    symptom_fields={"duration_days": 90, "symptom_type": ["sore that won't heal"]},
)
print(result["condition"], "|", result["severity_label"], "|", result["triage_level"], "| embedding_dim:", len(result["text_embedding"]))

result2 = predict(
    "There's a small brown mole on my arm, same size and color for years, never bothered me",
    symptom_fields={"duration_days": 900, "spreading": False},
)
print(result2["condition"], "|", result2["severity_label"], "|", result2["triage_level"])

## Step 11 — Download the outputs

In [ ]:
files.download('symptom_classifier_biobert.pkl')
files.download('interface_metadata.json')
files.download('evaluation_report.txt')